# Fake News Detection – Exploratory Data Analysis

Quick EDA on the Kaggle Fake News dataset (or the bundled sample).

We look at:
1. Class balance (real vs fake)
2. Text length distribution
3. Top n-grams per class
4. Missing values

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from preprocess import clean_series

sns.set_theme(style='whitegrid')
DATA = pathlib.Path('..') / 'data'

if (DATA/'True.csv').exists() and (DATA/'Fake.csv').exists():
    real = pd.read_csv(DATA/'True.csv'); real['label'] = 0
    fake = pd.read_csv(DATA/'Fake.csv'); fake['label'] = 1
    df = pd.concat([real, fake], ignore_index=True).sample(frac=1, random_state=42)
else:
    df = pd.read_csv(DATA/'fake_news.csv')
print(df.shape)
df.head()

## 1. Class balance

In [ ]:
ax = sns.countplot(x='label', data=df, palette=['#2ecc71', '#e74c3c'])
ax.set_xticklabels(['Real (0)', 'Fake (1)'])
ax.set_title('Class distribution')
plt.show()

## 2. Missing values

In [ ]:
df.isna().sum()

## 3. Text length distribution

In [ ]:
df['len'] = df['text'].fillna('').str.split().str.len()
sns.histplot(data=df, x='len', hue='label', bins=40, kde=True, palette=['#2ecc71', '#e74c3c'])
plt.title('Article length (words) by class')
plt.show()

## 4. Top uni-grams per class

In [ ]:
df['clean'] = clean_series(df['text'])

def top_terms(corpus, n=15):
    cv = CountVectorizer(max_features=2000)
    X = cv.fit_transform(corpus)
    freqs = X.sum(axis=0).A1
    return pd.Series(freqs, index=cv.get_feature_names_out()).nlargest(n)

real_top = top_terms(df.loc[df.label==0, 'clean'])
fake_top = top_terms(df.loc[df.label==1, 'clean'])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
real_top.sort_values().plot.barh(ax=axes[0], color='#2ecc71', title='Top words – REAL')
fake_top.sort_values().plot.barh(ax=axes[1], color='#e74c3c', title='Top words – FAKE')
plt.tight_layout(); plt.show()

## ✅ Next step

Run `python ../train.py` to fit the three classifiers and save them under
`../models/`, then launch the Streamlit app with `streamlit run ../app.py`.